<div style="background: linear-gradient(135deg, #1a1a2e 0%, #2d0a16 100%); padding: 32px; border-radius: 12px; text-align: center; font-family: monospace;">
<h1 style="color: #FFD700; font-size: 2.6em; margin: 0; letter-spacing: 3px;">🌿 THE NEIGHBOR GAMES 🌿</h1>
<h2 style="color: #ffffff; font-size: 1.2em; margin: 10px 0 0 0; font-weight: normal; letter-spacing: 1px;">k-Nearest Neighbor Prediction Championship</h2>
<p style="color: #aaaacc; margin: 10px 0 0 0; font-size: 0.9em;">Elements of Data Science · Temple University</p>
<p style="color: #FFD700; margin: 8px 0 0 0; font-size: 0.85em; font-style: italic;">"May the best features be ever in your favor."</p>
</div>

---

### 🏆 Competition Overview

| Round | Task | Challenges |
|---|---|---|
| 🏋️ Training Camp | Build k-nearest neighbors **by hand**, one line at a time, for one mystery molecule | 1–6 |
| 🌿 Tribute Round | Shared baseline: run as-is | — |
| ⚔️ Arena Round | Tune `k` and the features | 7 |
| 🎯 Prize Round | Predict the bp of two real molecules | 8–9 |
| 🌟 Wildcard | Inverse-distance weighting | 10 |

**How the challenges work.** Each 🖊️ challenge is **one line of code** with `...` where your team fills in the missing piece. Run it, then run the ✅ check cell under it: it tells you whether you got it, and gives a hint if not. Work it out together; everyone on the team fills in their own notebook.

After each round, click **📡 Submit** to send your team's score to the live scoreboard. Submissions open when your instructor turns the light **🟢 green**.

---
### ⚙️ Setup — run this first

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')

import sys, os
# Find the course tools (Fall 2026/shared-public), wherever this notebook sits in the course folder
for _p in ['../shared-public', '../../shared-public', '../../../shared-public', '/home/jovyan/data/Fall 2026/shared-public']:
    if os.path.exists(os.path.join(_p, 'class_games.py')):
        sys.path.insert(0, os.path.abspath(_p))
        break
from class_games import submit_button, neighbor_team, neighbor_check

def make_submit_button(round_name, score_var_name, extra_fn=None):
    "A Submit button for one round; it sends the current value of score_var_name under team_name."
    submit_button(round_name, score_var_name, extra_fn, game='neighbor_games')

Temple_color = '#9E1B34'
Gold_color   = '#FFD700'
print("✅ Ready")

---
### 📛 Your team
Pick a team name, and list **everyone on your team** (first name and last initial) in `team_members`: each name in quotes, separated by commas. Everyone on the team uses the **same team name**. Add or delete `"..."` entries to match your team size.

In [ ]:
team_name = "..."   # e.g. "District 4: The Alkanol Alliance"

In [ ]:
team_members = ["...", "...", "..."]   # e.g. ["Ada L.", "Grace H.", "Alan T."]

Run this to register your team with the class scorecard. It works before the competition starts.

In [ ]:
neighbor_team()

---
### 📂 Data & Competition Split
72 alcohols with their boiling points (`bp`, in K). Three quarters of them are the **training** set, the molecules we know. The other quarter is the **test** set: we pretend not to know their boiling points, predict them, then check.

In [ ]:
ROH_data = Table().read_table('data/ROH_data.csv')
np.random.seed(42)
shuffled  = ROH_data.sample(with_replacement=False)
split_n   = int(0.75 * shuffled.num_rows)
train_raw = shuffled.take(np.arange(split_n))
test_raw  = shuffled.take(np.arange(split_n, shuffled.num_rows))
print(f"Training: {train_raw.num_rows} rows  |  Test: {test_raw.num_rows} rows")
train_raw.select('bp', 'MW', 'degree', 'carbons').show(5)

---
### 📏 Standardization
Molecular weight runs from about 30 to 200 g/mol, but degree is just 1, 2 or 3. Without rescaling, MW would swamp every distance. So we put every feature in **standard units**:

$$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

> ⚠️ Compute $\mu$ and $\sigma$ from the **training data only**, then apply them to both sets ([Inferential Thinking 14.2](https://inferentialthinking.com/chapters/14/2/Variability.html)).

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 1 of 10 — The z-score formula</strong><br>
Finish the one-line function: subtract the mean, then divide by the SD.
</div>

In [ ]:
def z(x, mu, sigma): return (... - ...) / ...

In [ ]:
neighbor_check('z')

Your `z` now standardizes every feature, using the training mean and SD for both sets.

In [ ]:
all_features = ["MW", "degree", "carbons"]
train_stats = {f: (np.mean(train_raw.column(f)), np.std(train_raw.column(f))) for f in all_features}

train = train_raw.select('bp')
test  = test_raw.select('bp')
for f in all_features:
    mu, sigma = train_stats[f]
    train = train.with_columns(f, z(train_raw.column(f), mu, sigma))
    test  = test.with_columns(f, z(test_raw.column(f), mu, sigma))

print(f"{'Feature':<10} {'train mean':>11} {'train SD':>9}   (should be 0 and 1)")
for f in all_features:
    print(f"{f:<10} {np.mean(train.column(f)):>11.3f} {np.std(train.column(f)):>9.3f}")

---
## 🏋️ Training Camp — k-nearest neighbors by hand
Before the competition, your team builds the whole method for **one mystery molecule**: the first molecule in the test set. Its boiling point is hidden until the end.

The idea ([Inferential Thinking 17.4](https://inferentialthinking.com/chapters/17/4/Implementing_the_Classifier.html)): find the training molecules **closest** to the mystery molecule, and predict that it boils at about the **average** of theirs.

In [ ]:
features = ["MW", "degree"]
target   = ["bp"]
mystery  = np.array(test.select(features).row(0))   # the mystery molecule, in standard units

print("Mystery molecule:", {f: test_raw.column(f).item(0) for f in features}, "  bp = ???")
print("In standard units:", {f: round(float(v), 3) for f, v in zip(features, mystery)})

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 2 of 10 — The distance formula</strong><br>
The straight-line distance between two points is the square root of the sum of the squared differences:<br>
&nbsp; distance = √( Σ (pt1 − pt2)² )<br>
<code>pt1 - pt2</code> subtracts the arrays element by element, one difference per feature.
</div>

In [ ]:
def distance(pt1, pt2): return np.sqrt(np.sum((... - ...) ** 2))

In [ ]:
neighbor_check('distance')

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 3 of 10 — One distance</strong><br>
How far is the <strong>first training molecule</strong> from the mystery molecule? The first argument is filled in; give the second point.
</div>

In [ ]:
d0 = distance(np.array(train.select(features).row(0)), ...)

In [ ]:
neighbor_check('d0')

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 4 of 10 — Every distance</strong><br>
Now do the same for <strong>every</strong> training molecule, in one line. <code>train.select(features).rows</code> goes through the training molecules one row at a time.
</div>

In [ ]:
all_dists = [distance(np.array(row), ...) for row in train.select(...).rows]

In [ ]:
neighbor_check('all_dists')

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 5 of 10 — The 5 nearest neighbors</strong><br>
Add the distances to the training table as a column, <strong>sort</strong> by it (closest first), and <strong>take</strong> the first 5 rows.
</div>

In [ ]:
nearest = train.with_column('Distance', ...).sort(...).take(np.arange(5))

In [ ]:
neighbor_check('nearest')

In [ ]:
nearest.show()

**See the neighbors.** Every training molecule in standard units: the ★ is the mystery molecule, circled points are its 5 nearest neighbors. (Degree is 1, 2 or 3, so the points sit in three rows; a little jitter keeps them from hiding each other.)

In [ ]:
np.random.seed(0)
jitter = np.random.uniform(-0.04, 0.04, train.num_rows)
plt.figure(figsize=(7, 4.5))
plt.scatter(train.column(features[0]), train.column(features[1]) + jitter, c=train.column('bp'), cmap='viridis', s=45, label='training molecules')
plt.colorbar(label='bp (K)')
plt.scatter(nearest.column(features[0]), nearest.column(features[1]), s=260, facecolors='none', edgecolors=Temple_color, lw=2.5, label='5 nearest')
plt.scatter(mystery[0], mystery[1], marker='*', s=420, color=Gold_color, edgecolors='black', zorder=5, label='mystery molecule')
plt.xlabel(f'{features[0]} (standard units)'); plt.ylabel(f'{features[1]} (standard units)')
plt.title('The mystery molecule and its 5 nearest neighbors')
plt.legend(loc='upper left', fontsize=9); plt.tight_layout(); plt.show()

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 6 of 10 — The prediction</strong><br>
Predict the mystery molecule's bp as the <strong>average</strong> bp of its 5 neighbors.
</div>

In [ ]:
prediction = np.average(nearest.column(...))

In [ ]:
neighbor_check('prediction')

### 🔧 Your toolkit
That's the whole method. The functions below are **the same lines you just wrote**, packaged so they work for any test molecule (`row`) and any `k`. The competition rounds use them.

In [ ]:
def distances(training, point, features):
    "Challenge 4: the distance from point to every training molecule, as a new column."
    return training.with_column('Distance', [distance(np.array(r), point) for r in training.select(features).rows])

def closest(training, point, k, target, features):
    "Challenge 5: the k nearest neighbors."
    return distances(training, point, features).sort('Distance').take(np.arange(k))

def predict_knn(row, train, test, k=5):
    "Challenge 6: the average target value of the k nearest neighbors of test molecule number `row`."
    point = np.array(test.select(features).row(row))
    return np.average(closest(train, point, k, target, features).column(target[0]))

def predict_knn_weighted(row, train, test, k=5):
    "Wildcard: like predict_knn, but closer neighbors count more (Challenge 10)."
    point = np.array(test.select(features).row(row))
    near = closest(train, point, k, target, features)
    weights = 1 / (near.column('Distance') + 1e-9)
    return np.sum(near.column(target[0]) * weights) / np.sum(weights)

def compute_score(predictions, label='Score'):
    "RMSE: the typical size of a prediction error, in K. Lower is better."
    actual = test.column('bp')
    preds  = np.array(predictions)
    if len(preds) != len(actual):
        print(f"⚠️  Need {len(actual)} predictions, got {len(preds)}")
        return None
    rmse = np.sqrt(np.mean((actual - preds) ** 2))
    fill = int(max(0, 28 - rmse / 2))
    print(f"{'─'*52}\n  {label}\n  RMSE = {rmse:.3f} K   [{'█'*fill}{'░'*(28-fill)}]\n{'─'*52}")
    return round(rmse, 3)

In [ ]:
neighbor_check('toolkit')

---
## 🌿 Tribute Round — Run exactly as written
Now predict **every** test molecule, with molecular weight only and k = 5, and score the predictions.

In [ ]:
features = ["MW"]
k = 5
tribute_predictions = [predict_knn(i, train, test, k=k) for i in np.arange(test.num_rows)]
tribute_rmse = compute_score(tribute_predictions, label="Tribute Baseline (MW only, k=5)")

In [ ]:
actual_bp = test.column('bp')
plt.figure(figsize=(6,5))
plt.scatter(actual_bp, tribute_predictions, color=Temple_color, alpha=0.7, edgecolors='white', s=60)
diag = np.linspace(min(actual_bp), max(actual_bp), 100)
plt.plot(diag, diag, '--', color=Gold_color, lw=1.5, label='Perfect')
plt.xlabel('Actual bp (K)'); plt.ylabel('Predicted bp (K)')
plt.title(f'Tribute  |  RMSE = {tribute_rmse:.2f} K')
plt.legend(); plt.tight_layout(); plt.show()

#### 📡 Submit your Tribute score

In [ ]:
make_submit_button('Tribute', 'tribute_rmse')

---
## ⚔️ Arena Round — Tune Your Model!
**Choose your settings.** Available features: `"MW"`, `"degree"`, `"carbons"` (all standardized). Try different combinations and values of `k`; rerun Challenge 7 after each change. Your **best** Arena score counts.

In [ ]:
features = ["MW", "degree"]   # try adding "carbons"
k = 5                         # try 1, 3, 7, 10, 15 …

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 7 of 10 — Predict every test molecule</strong><br>
Fill in the list comprehension. <code>i</code> is the row number of each test molecule; <code>predict_knn(row, train, test, k)</code> predicts one of them.
</div>

In [ ]:
arena_predictions = [predict_knn(..., train, test, k=k) for i in np.arange(...)]

In [ ]:
neighbor_check('arena')

In [ ]:
arena_rmse = compute_score(arena_predictions, label=f"Arena: features={features}, k={k}")

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(actual_bp, arena_predictions, color='steelblue', alpha=0.7, edgecolors='white', s=60)
plt.plot(diag, diag, '--', color=Gold_color, lw=1.5, label='Perfect')
plt.xlabel('Actual bp (K)'); plt.ylabel('Predicted bp (K)')
plt.title(f'Arena  |  k={k}, features={features}\nRMSE = {arena_rmse:.2f} K')
plt.legend(); plt.tight_layout(); plt.show()

**Scouting report: which k?** This sweep tries k = 1 to 20 with your current features. Small k follows single neighbors (noisy); large k averages over molecules that aren't really alike.

In [ ]:
k_values = np.arange(1, 21)
rmse_list = make_array()
for kk in k_values:
    preds = [predict_knn(i, train, test, k=kk) for i in np.arange(test.num_rows)]
    rmse_list = np.append(rmse_list, np.sqrt(np.mean((test.column('bp') - np.array(preds)) ** 2)))
best_k = k_values.item(np.argmin(rmse_list))
plt.figure(figsize=(7,4))
plt.plot(k_values, rmse_list, 'o-', color=Temple_color, lw=2, ms=6)
plt.axvline(best_k, color=Gold_color, ls='--', label=f'Best k={best_k}')
plt.xlabel('k'); plt.ylabel('Test RMSE (K)')
plt.title(f'k vs RMSE  |  features={features}')
plt.legend(); plt.tight_layout(); plt.show()
print(f"Best k={best_k}  RMSE={min(rmse_list):.3f} K")

#### 📡 Submit your Arena score

In [ ]:
make_submit_button('Arena', 'arena_rmse',
    extra_fn=lambda: {'features': features, 'k': int(k)})

---
## 🎯 Prize Round — Predict Real Molecules!

| Molecule | MW (g/mol) | Degree | Carbons | Actual bp |
|---|---|---|---|---|
| **Ethanol** | 46.07 | 1 | 2 | 351.4 K |
| **Isopropanol (IPA)** | 60.10 | 2 | 3 | 355.4 K |

A new molecule has to be standardized with the **training** mean and SD, exactly like the training data. That's why `standard_units()` won't do: it would use the mean and SD of just this one molecule.

In [ ]:
features = ["MW", "degree"]   # ← your best features from the Arena
k = 5                         # ← your best k

ethanol_raw = {"MW": 46.07, "degree": 1, "carbons": 2}
ipa_raw     = {"MW": 60.10, "degree": 2, "carbons": 3}

# Ethanol, the worked example: z(raw value, training mean, training SD) for each feature
ethanol_std = np.array([z(ethanol_raw[f], train_stats[f][0], train_stats[f][1]) for f in features])
print("Ethanol standardized:", {f: round(float(v), 3) for f, v in zip(features, ethanol_std)})

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 8 of 10 — Standardize isopropanol</strong><br>
Same pattern as ethanol. <code>train_stats[f]</code> holds the training (mean, SD) of feature <code>f</code>: <code>train_stats[f][0]</code> is the mean, <code>train_stats[f][1]</code> the SD.
</div>

In [ ]:
ipa_std = np.array([z(ipa_raw[f], ..., ...) for f in features])

In [ ]:
neighbor_check('ipa_std')

In [ ]:
ethanol_neighbors = closest(train, ethanol_std, k, target, features)
ethanol_bp_pred = np.average(ethanol_neighbors.column('bp'))
ethanol_error = abs(ethanol_bp_pred - 351.4)
print(f"Ethanol     predicted = {ethanol_bp_pred:.1f} K   actual = 351.4 K   error = {ethanol_error:.1f} K")

ipa_neighbors = closest(train, ipa_std, k, target, features)
ipa_bp_pred = np.average(ipa_neighbors.column('bp'))
ipa_error = abs(ipa_bp_pred - 355.4)
print(f"Isopropanol predicted = {ipa_bp_pred:.1f} K   actual = 355.4 K   error = {ipa_error:.1f} K")

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 9 of 10 — The Prize score</strong><br>
The Prize score is the <strong>average</strong> of the two errors.
</div>

In [ ]:
prize_score = (ethanol_error + ipa_error) / ...

In [ ]:
neighbor_check('prize_score')

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
x, w = np.arange(2), 0.3
ax.bar(x-w/2, [351.4, 355.4], w, label='Actual', color=Gold_color, edgecolor='black', lw=0.5)
ax.bar(x+w/2, [ethanol_bp_pred, ipa_bp_pred], w, label='Predicted', color=Temple_color, edgecolor='black', lw=0.5, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(['Ethanol\n(351.4 K)', 'IPA\n(355.4 K)'], fontsize=11)
ax.set_ylabel('Boiling Point (K)'); ax.set_title(f'Prize Round: {team_name}  ·  score {prize_score:.2f} K')
ax.legend(); plt.tight_layout(); plt.show()

#### 📡 Submit your Prize score

In [ ]:
make_submit_button('Prize', 'prize_score')

---
## 🌟 Wildcard Round — Inverse-Distance Weighting
In plain kNN, all 5 neighbors get an equal vote, even one that's much farther away than the others. **Weighting** gives each neighbor a vote of 1 / distance, so the closest ones count most. Back to the mystery molecule's `nearest` table from the Training Camp:

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 10 of 10 — The weights</strong><br>
One weight per neighbor: 1 divided by its distance. (The tiny <code>1e-9</code> avoids dividing by zero when a neighbor is at distance 0.)
</div>

In [ ]:
weights = 1 / (nearest.column(...) + 1e-9)

In [ ]:
neighbor_check('weights')

In [ ]:
weighted_prediction = np.sum(nearest.column('bp') * weights) / np.sum(weights)
print(f"Mystery molecule: plain average {prediction:.1f} K · weighted {weighted_prediction:.1f} K · actual {test.column('bp').item(0):.0f} K")

A neighbor at distance 0 has the same MW and degree as the mystery molecule: an isomer. Its weight is huge (1 / 0.000000001), so it essentially decides the weighted prediction by itself. Is that a good thing here?

Now the whole test set, weighted (`predict_knn_weighted` in your toolkit). Run as written.

In [ ]:
features = ["MW", "degree"]
k = 5
wildcard_predictions = [predict_knn_weighted(i, train, test, k=k) for i in np.arange(test.num_rows)]
wildcard_rmse = compute_score(wildcard_predictions, label=f"Wildcard (weighted): features={features}, k={k}")

#### 📡 Submit your Wildcard score

In [ ]:
make_submit_button('Wildcard', 'wildcard_rmse')

---
## 💬 Reflection

1. **In Challenge 2, why do we square the differences before adding them?** What would go wrong if we just added `pt1 - pt2`?
2. **Why do we standardize with the training mean and SD, and apply the same ones to the test set and to new molecules?**
3. **What does `i` represent in the list comprehension in Challenge 7?**
4. **Did adding more features improve your Arena RMSE? Did a bigger k always help?**
5. **When would weighting by 1 / distance help most?**

*Your answers here:*

1.

2.

3.

4.

5. 